In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# RLLM-6: GRPO and its refinements

## Where this session fits

> **The limitation we start from: the plain GRPO of Intro session C2 gives us two reasons to look further: a reported gap in results, and weak spots visible in its objective.**

- **The question:** which changes to plain GRPO do later papers report, and on what evidence?
- **Position: session 6 of the RL for LLMs series (RLLM-1 to RLLM-14)**
    - RLLM-4 gave PPO in depth, including what its critic (a second large network) costs
    - RLLM-5 ran plain GRPO, with no critic, on reasoning (RLVR): this session refines it
    - RLLM-7 begins part 2 of the RLLM series, RFT Alternative methods (**RFT**, defined in RLLM-1)
    - [RLLM-6A](RLLM_6A_GRPO_Refinements_Appendix.ipynb) is an optional appendix: detail and evidence for this session
- Reward sources: people compare (RLHF) (RLLM-2); an AI compares (RLAIF) (RLLM-3); a program checks (RLVR) (RLLM-5) (this session refines the optimizer that consumes the reward) <!-- long-ok -->
- **Series goal:** turn a pretrained language model into a useful, aligned assistant with reinforcement learning, and know the methods well enough to choose among them

**By the end you can**

1. give two reasons to look beyond plain GRPO, and say how far each one goes
2. name each part of the GRPO objective that a later recipe changes, and say why
3. compute a token's **share of the loss** under two averaging rules, and an advantage with and without the standard deviation
    - the share of the loss is how much that token's term counts in the objective
4. say what each change risks, and which are arithmetic, which are arguments and which are **design choices** (a choice with no proof)
5. say what GRPO gives up compared with PPO, and name **the three axes** that part 2 varies


## Reasons: two reasons to look beyond plain GRPO

> **Neither reason is proof alone. Together they say where to look; the later evidence says what helped.**

- **Reason 1, a reported gap:** the DAPO paper's plain GRPO run on a 32-billion-parameter base model scored 30 on AIME 2024
    - DeepSeek reported 47 for DeepSeek-R1-Zero-Qwen-32B
    - **AIME 2024:** competition math with exact numeric answers, averaged over 32 samples
    - **caveat:** the 30 is DAPO's own reimplementation, and DeepSeek's recipe is not public: the gap may come from undisclosed details
- **Reason 2, weak spots in the objective itself** (our reading): each is a claim, tested later
    - length averaging, advantage scale, rare-token clip, wasted groups, **cut-offs** (completions stopped by the length limit)
- the papers: **DAPO** (an open recipe, 50 on the same benchmark), **Dr. GRPO** ("GRPO Done Right"), **GSPO** (Group Sequence Policy Optimization); not comparable with RLLM-5's numbers <!-- long-ok -->
- next: to change an objective we must name its parts


## Objective: the GRPO parts, named first

**Why this section?** To change an objective we must name its parts. C2 derived it, RLLM-5 ran it; this is the recall.

- **group:** $G$ completions sampled for one prompt (C2)
- **advantage:** reward minus the group mean, divided by the group standard deviation: one number per completion (C2)
- **ratio** (the **importance ratio**): new over old probability of a token
    - the factor that corrects for reusing samples from the old policy (B4)
- **clip:** limits the ratio to $1 \pm \epsilon_{\text{clip}}$ (B4)
- **KL term:** keeps the policy near the reference model, coefficient $\beta$ (C1)
- **averaging:** a mean over the tokens of a completion ($1/\lvert \y_i \rvert$), then over the group ($1/G$)

**The same parts, in one formula**

> **Per token: PPO's clipped term, using the completion's one advantage, minus a KL term; averaged over the tokens, then over the group.**

$$
\frac{1}{G} \sum_{i=1}^{G} \frac{1}{\lvert \y_i \rvert} \sum_{\tt} \Bigl[ \min{} \bigl( r_{i,\tt}(\theta)\, \hat{\advseq}_i, \;\; \text{clip}\bigl(r_{i,\tt}(\theta),\, 1 - \epsilon_{\text{clip}},\, 1 + \epsilon_{\text{clip}}\bigr)\, \hat{\advseq}_i \bigr) \; - \; \beta \, \KL_{i,\tt} \Bigr]
\qquad \hat{\advseq}_i = \frac{\rew_i - \text{group mean of } \rew}{\text{group standard deviation of } \rew}
$$

- subscripts $i,\tt$, completion first, are the GRPO paper's order (C2), which wins on a clash
- $\KL_{i,\tt}$ is GRPO's per-token estimate of the KL (C2)
    - RLLM-4 used the log-ratio as a PPO reward


## Map: seven changes, in six sections

**Why this table?** It is the map of the session: one row per change, and what kind of reason backs it.

- **Cut-off:** a completion stopped by the length limit
- **Baseline:** what is subtracted from the reward to form the advantage


| Section | Plain GRPO (C2) | Change (source) | Kind of reason |
|:--|:--|:--|:--|
| Averaging | per completion, then group | count all tokens, or divide by a constant (DAPO, Dr. GRPO) | arithmetic, then argument |
| Advantage scale | group mean and std | drop the std (Dr. GRPO) | argument |
| Clip | per token, same width both sides | clip higher: a wider upper edge (DAPO) | design choice |
| Clip | per token | one ratio per completion (GSPO) | argument |
| Group filtering | every group used | filter all-equal groups (DAPO) | arithmetic, ablation |
| Cut-offs | scored as they stand | mask, or penalize gently (DAPO) | design choice, ablation |
| KL term | a term in the loss (PPO: a reward at every token) | drop it (DAPO, Dr. GRPO) | design choice |

> **A menu, not a recipe, except one bundle: DAPO's (token-level averaging, clip higher, group filtering, cut-offs, no KL) was run together and answers reason 1. Each change is also a claim under reason 2.**

- **combine:** the DAPO bundle
    - dropping the std (Dr. GRPO) fits with it: DAPO leaves that open
- **conflict:** two rules for one knob
    - Dr. GRPO's constant divisor against token-level counting (Averaging)
    - GSPO's completion ratio against clip higher (one clips completions, one clips tokens)
- the baseline (the critic's value in PPO, the group mean in GRPO) does not change: the Credit section returns to that


## Averaging: how the loss is averaged

> **GRPO divides by the completion's own length first, so each token of a long completion counts less than each token of a short one.**

- **the knob:** how much each token's term counts in the loss
- **what we change:** count all tokens of the group alike (DAPO), or divide by a constant (Dr. GRPO)
- **believe it?** the sizes are arithmetic; why they hurt is each paper's argument

<img src="images/rllm6_weights.png" width=58%>

**The sizes, and why DAPO says it matters**

- a group of $G = 2$ completions, of $10$ and $100$ tokens: a token's **share** is $1/(G \lvert \y_i \rvert)$
    - GRPO: 0.0500 against 0.0050, 10 : 1 (computed here)
    - token-level: 0.0091 each; both rules give the group a total weight of $1$
- **DAPO** (its argument): good long reasoning is learned slowly, long gibberish is penalized little
- DAPO's ablation: the least accuracy, but more stable training (paper's words; RLLM-6A)
- Dr. GRPO's separate argument, about wrong answers drifting longer, is in RLLM-6A


## Advantage scale: whether to divide by the standard deviation

> **Dr. GRPO argues that dividing by the group standard deviation weighs almost-all-wrong and almost-all-right questions more, relative to the undivided weight.**

- **the knob:** whether to divide the reward gap by the group standard deviation
- **what we change:** drop the division (Dr. GRPO), while DAPO and GSPO keep it: an open design choice
- **believe it?** the amplification is arithmetic; that it harms is an argument

<img src="images/rllm6_advscale.png" width=40%>

- group of $8$, rewards in $\{0, 1\}$: a lone success gets 2.47 divided, 0.88 undivided
    - a $4$-of-$8$ question gets 0.94 and 0.50
    - lone success over balanced: 2.65 divided, 1.75 undivided, a ratio of ratios of 1.5 (sample std, dividing by $G - 1$)
- the argument's premise (ours): both questions are equally informative, so equally weighted
    - against it: a lone success is rarer, so it could deserve more


## Clip: a higher upper bound

<!-- seed-pay: clip-heuristic -->
> **From RLLM-4: the clip is a heuristic brake, not a bound on the KL, so its shape is a design choice. The upper clip matters only for tokens with a positive advantage.**

- **the knob:** the width of the clip on the ratio (new over old probability)
- **what we change:** a higher upper edge: $\epsilon_{\text{high}} = 0.28$ against $\epsilon_{\text{low}} = 0.2$ (DAPO)
- **believe it?** the mechanism is our reading; the gain rests on DAPO's ablation

<img src="images/rllm6_clip.png" width=52%>

**Why a relative cap narrows exploration**

- the clip point is relative: with $\epsilon_{\text{clip}} = 0.2$ and a positive advantage, the push stops at $1.2 \times$ the old probability
    - $0.01 \to 0.012$; $0.9 \to 1.08$, impossible: a probability stops at $1$
    - the clip removes that token's gradient; it is not a hard cap (RLLM-4)
- a rare token's push stops after $0.002$, a likely one's after $0.1$ (figure): rare tokens explore, and grow slowly
- over many updates the distribution narrows: **entropy collapse**, observed by the paper (the mechanism is our reading)
- the paper keeps $\epsilon_{\text{low}}$ low: raising it would push low-probability tokens toward zero
- **risk:** tuned in one setting
- next: the same clip, but with the ratio taken on the whole completion


## Clip: one ratio per completion

> **GSPO defines the importance ratio, and its clip, on the whole completion instead of on each token.**

- **the knob:** the unit the ratio and clip act on: a token, or a completion
- **what we change:** one ratio per completion, the geometric mean of the token ratios (GSPO's "sequence ratio")
- **the argument:** the reward belongs to the whole completion, but one token gives a noisy weight. The completion ratio averages that noise
- **believe it?** the paper's argument, tested in its own models
    - our figure shows a mechanism, not a measurement
    - the 4 outside tokens are not clipped, so their gradients are kept

<img src="images/rllm6_ratio.png" width=52%>

- the formula and GSPO's real clip ranges are in RLLM-6A


## Group filtering: do not waste groups

<!-- seed-pay: all-equal-group -->
> **From RLLM-5 and C2: if every completion in a group earns the same reward, all advantages are zero and the group contributes no gradient.**

- **the knob:** which groups enter the update
- **what we change:** **dynamic sampling** (DAPO): keep a prompt only if its group has a success and a failure
    - sample until the batch is full
- **believe it?** the waste is arithmetic; the benefit is DAPO's ablation

<img src="images/rllm6_dynamic.png" width=46%>

- a prompt answered correctly with probability $q$ gives an all-equal group of $8$ with probability $q^8 + (1-q)^8$: 0.008 at $q=0.5$, 0.43 at $q=0.9$
- as the policy improves, more prompts reach the right-hand end (our reading)
- the paper notes more generation per step, but fewer steps


## Cut-offs: do not punish them unfairly

> **A cut-off completion may get a punishing reward even if its reasoning was sound; the DAPO paper says this adds noise.**

- **the knob:** the reward of a completion stopped by the length limit
- **what we change:** **overlong filtering** (mask its loss) or **soft overlong punishment** (a penalty that grows only near the limit)
- **believe it?** design choices; the evidence is DAPO's ablation (RLLM-6A), not a proof
- under plain GRPO a cut-off has no final answer, so it fails the accuracy check, and usually the format check (RLLM-5)
- the paper's soft setting: no penalty up to $16\,384$ tokens, then linear from $0$ to $-1$ over $4\,096$
    - generation stops at $20\,480$


## KL term: drop it?

> **Both papers set the KL coefficient to zero, for different reasons. Our reading: drop it with a verifier, think twice with a learned reward model.**

- **the knob:** the KL term, which keeps the policy near the reference model (C1, RLLM-4)
- **what we change:** set $\beta = 0$ (DAPO, Dr. GRPO)
- **believe it?** two arguments, no ablation number: a design choice
- **DAPO:** a long-reasoning model should move far from its start, so the restriction is unnecessary
- **Dr. GRPO:** with a verifier (RLVR) there is no learned reward model to exploit, so no anchor is needed
- the benefit: no reference model in memory
- the risk: drift that the reward does not forbid (RLLM-6A has the rest)
- next: none of these seven changes touches how credit is shared among tokens


## Credit: what GRPO gives up
<!-- seed-pay: critic-two-roles -->

**Why this section?** The seven changes tune the objective. They leave the baseline alone, and the baseline is where PPO and GRPO differ.

> **From RLLM-4: the critic had two roles: a baseline, and credit assignment. The group mean replaces the first; GRPO gives up the second, since every token gets the same advantage.**

- the baseline is the role GRPO kept, so no change touches it
- none of the refinements restores the lost role
- next: where methods can differ on this, and how part 2 organizes them


## Axes: three ways methods differ

**Why this slide?** Samples, weights and credit are where methods differ; part 2 varies them one at a time.

- RLLM-7 teaches axis 1, RLLM-8 axes 2 and 3
- this is a preview of the questions, using three methods you know


> **Methods differ on three axes: whose samples, who supplies the weights, how fine the weights are.**

<img src="images/rllm6_axes.png" width=62%>

- **1. whose samples:** the model's own (a copy within a few updates is nearly the same) or another model's or a dataset's (RLLM-7)
- **2. who supplies the weights:** a reference to imitate (a dataset's token, a teacher), or a reward for the model's own completion (RLLM-8)
    - a **weight** is the multiplier an update puts on a token
- **3. how fine the weights are:** one number per completion, one per sampled token, or a distribution per token (RLLM-8)
- PPO and GRPO answer 1 and 2 alike and differ on 3: that is the credit gap
- not the Intro's "three choices" (weight, how far, baseline source) nor its "three questions"


## Hand-off: choosing a method after Part 1

**Why this slide?** The decks so far give three ways to get a reward. Before part 2 widens the options, here is the choice as Part 1 can make it.

> **What you hold decides the reward source. This version covers Part 1 options only; the full rule, with Part 2 options, comes in RLLM-13.**

- **a program checks the answer:** RLVR, reinforcement learning with verifiable rewards (RLLM-5)
- **only people can judge:** collect comparisons, train a reward model, run RLHF (RLLM-2)
- **an AI model can judge, guided by written principles:** RLAIF (RLLM-3)
- PPO or GRPO (in practice, our reading): RLVR recipes often start from GRPO
    - papers on repaired critics (VAPO, VC-PPO) argue PPO can win
- next: methods that change where the samples or the weights come from


# What we established, and what it sets up

- **Established**
    - two reasons to look beyond plain GRPO, neither proof alone: a reported gap, and weak spots in the objective
    - seven changes in six sections of the update
    - only the sizes in two (averaging, empty groups) are plain arithmetic
    - the rest are arguments or design choices from one paper's own setting: options, not a ranking
    - **DAPO:** token-level averaging, higher clip, dynamic sampling, overlong handling, no KL
    - **Dr. GRPO:** no length or std division, no KL
    - **GSPO:** one ratio per completion
- **Sets up**
    - **the limitation:** one scalar per completion, from the current policy's own samples, with no credit assignment
    - **RLLM-7, the start of part 2 (RFT Alternative methods):** varying the three axes
    - **which method to pick** (by what you hold): the Part 1 rule is the previous slide, the full rule is in RLLM-13
